# 01 · Generate switching dynamics

Generate x[t+1] = A[r[t]] x[t] + noise, with four state dimensions and three Markov regimes.

**Run cells in order.** The default smoke preset validates the pipeline; it is too small for a scientific conclusion.

## Setup

In [1]:
from pathlib import Path
import os, sys
ROOT = Path.cwd().resolve()
if not (ROOT / 'src' / 'twilight_moe').exists():
    ROOT = ROOT.parent
assert (ROOT / 'src' / 'twilight_moe').exists(), 'Start Jupyter from the project folder'
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from twilight_moe.config import load_config, run_dir
from twilight_moe.plotting import style, save_figure, COLORS
cfg = load_config()
style()
print('Preset:', cfg['name'], '| seeds:', cfg['seeds'])
print('Dataset root:', cfg['data_root'])

Preset: experiment | seeds: [7, 17, 27]
Dataset root: E:/HarunoKen/Twilight_MoE_Dynamics


## Inspect the generator

Regime r[t] acts on x[t]. The norm bound ensures stability even during switching.

In [2]:
from twilight_moe.dynamics import transition_matrices
A = transition_matrices()
print('Matrix operator norms:', np.linalg.norm(A, ord=2, axis=(1, 2)))
p = cfg['data']['stay_probability']
P = np.full((3, 3), (1-p)/2)
np.fill_diagonal(P, p)
display(pd.DataFrame(P, index=['r0', 'r1', 'r2'], columns=['r0', 'r1', 'r2']))

Matrix operator norms: [0.94 0.94 0.94]


,r0,r1,r2
r0,0.950,0.025,0.025
r1,0.025,0.950,0.025
r2,0.025,0.025,0.950


## Generate independent sequence splits

Independent random streams prevent window leakage. Existing datasets are checksum-verified.

In [3]:
from twilight_moe.data import ensure_data, load_data
paths = {seed: str(ensure_data(cfg, seed)) for seed in cfg['seeds']}
display(pd.Series(paths, name='Dataset path').to_frame())
data, normalization = load_data(cfg, cfg['seeds'][0])
display(pd.DataFrame({split: {'sequences': len(d['states']),
                            'transitions_per_sequence': d['regimes'].shape[1]}
                     for split, d in data.items()}).T)

,Dataset path
7,E:\HarunoKen\Twilight_MoE_Dynamics\generated\f...
17,E:\HarunoKen\Twilight_MoE_Dynamics\generated\4...
27,E:\HarunoKen\Twilight_MoE_Dynamics\generated\7...


,sequences,transitions_per_sequence
train,256,192
val,64,192
test,64,192


## Check stability and normalization

Normalization statistics are computed from training states only.

In [4]:
assert all(np.isfinite(d['states']).all() for d in data.values())
assert np.allclose(normalization['mean'], data['train']['states'].mean(axis=(0, 1)))
display(pd.DataFrame(normalization, index=[f'x{i}' for i in range(4)]))

,mean,std
x0,-0.000522,0.232311
x1,0.001562,0.232734
x2,-0.000648,0.235849
x3,-0.001466,0.235677


## Provenance

The E: metadata folder records generator parameters, seed, matrices, regime alignment, and SHA-256 hashes. No external dataset is used in version 1.